# 零、总述

随着学习的深入，我们开始使用主要针对**文本生成**进行训练的模型，这类模型通常被称为**生成式预训练的Transformer(Generativ Pre-trained Transformer, GPT)。** 这些模型具有非常强大的能力，可以根据用户提供的提示词生成文本。通过**提示词工程（Prompt Engineering）**，我们可以更合理的设计这些提示词，从而提高模型生成文本的质量。

在本章中，我们将更加深入地探索这些生成式模型，并进一步学习**提示词工程、使用生成式模型进行推理、结果验证，以及对模型输出进行评估** 等内容。

# 一、使用文本生成模型

在开始学习**提示词工程**的基础知识之前，我们首先需要了解如何使用一个文本生成模型。我们应该怎样选择要使用的模型呢？是选择闭源模型（Proprietary）还是开源模型？

这些问题将作为我们使用文本生成模型的起点

## 1.1 选择 Text Generation Model

我们以选择闭源和开源模型来开始选择文本生成模型，即使专有模型的性能会很好，但是为了学习使用，我们使用开源模型。

那么，我们使用 `Phi-3-mini` 模型，它有 3.8B（38亿）的参数量，特别适合在 8G 的 VRAM 上运行。

总的来说，将小模型变成大模型要比从大模型变成小模型要容易得多。更小的模型会提供更好的引导，而且可以为后续大模型的学习会打下坚实的基础

## 1.2 加载文本生成模型

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 加载模型的分词器
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi3-mini-4k-instruct")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

还记得我们第一个notebook中的示例么，我们让他讲一个笑话，我们依旧使用这个例子

In [ ]:
# 提示词
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

在内部，`transformers.pipeline` 首先会将我们的 messages 转换成特定的提示词模板，我们可以使用下述的函数看一下模型将我们的 messages 转成了什么：

In [ ]:
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

在输出结果中，你可以看到有两个特殊的词元：`<|user|>` 和 `<|assistant|>`，这个提示词模版的说明，如下图所示

<center>
<img src="./resources/phi3-template.png">
</center>

## 1.3 控制模型输出

除了提示词工程之外，我们可以调整模型的参数，比如：`temperature` 和 `top_p`.

这些参数会控制输出的随机性。LLM 每次生成一个 token 时，每一个可能的 token 都会被分配一个似然值。

当我们加载模型时，设置 `do_sample=False` 的目的是为了确保它生成的内容有一些连贯性，它意味着每次生成的 token 都是最可能的那一个词。然而，为了使用 `temperature` 和 `top_p` 参数，我们将设置 `do_sample=True`。

1. temperature

`temperature` 参数控制文本生成的随机性和多样性。它定义了有多大的可能会选择小概率的 token。更高的 `temperature` 通常会导致更*多样化*的输出，而更低的 `temperature` 会生成更稳定的输出。请注意，即使你设置了相同的温度值，多次运行下述代码结果也是会变化的，因为 `temperature` 引入了随机选择的行为

In [ ]:
# 更高的温度
output1 = pipe(messages, do_sample=True, temperature=0.8)
print(output1[0]["generated_text"])

In [ ]:
# 更低的温度
output2 = pipe(messages, do_sample=True, temperature=0.2)
print(output2[0]["generated_text"])

2. top-p

`top-p`：也叫做**核采样**，它是一种采样的技术，来控制 LLM 来选择哪些 token 的子集，这些 token 的子集就被称为 nucleus(核)。它会选择那些直到累积的概率达到其设定的值的 token，所以，如果将 `top_p` 设置为 1，则它会选择所有 token。`top_p` 对模型生成的 token 的影响如下：

<center>
<img src="./resources/top_p.png">
</center>


In [ ]:
output = pipe(messages, do_sample=True, top_p=0.6)
print(output[0]["generated_text"])

# 二、介绍提示词工程

## 2.1 提示词的基本成分

## 2.2 基于指令的提示词

# 三、高级的提示词工程

## 3.1 提示词潜在的复杂度

## 3.2 上下文内学习：提供的例子

## 3.3 提示词链：将问题分解

# 四、使用生成式模型进行推理

## 4.1 CoT（Chain-of-Thought）: 在回答之前思考

## 4.2 Self-Consistency(自洽性)：样例输出

## 4.3 Tree-of-Thought（思维树）: 探索内部的步骤

# 五、输出验证

## 5.1 提供示例

## 5.2 语法：约束后的样例